# Putting the G(eneration) in RAG

In [ ]:
from openai import OpenAI
from pinecone import Pinecone

# If you've run 04_rag_retrieval.ipynb in the same environment,
# you should already have an active Pinecone index named
# according to INDEX_NAME and an OpenAI client initialized.
# Run the following and fill in your API keys if needed.

pinecone_key = ""
client = OpenAI(api_key="")
pc = Pinecone(api_key=pinecone_key)
index = pc.Index(name='semantic-search-rag')
ENGINE = 'text-embedding-3-small'


In [ ]:
def get_embedding(text, engine=ENGINE):
    resp = client.embeddings.create(input=[text], model=engine)
    return resp.data[0].embedding


def query_from_pinecone(query, top_k=3, include_metadata=True):
    embedding = get_embedding(query, engine=ENGINE)
    return index.query(vector=embedding, top_k=top_k, namespace='default', include_metadata=include_metadata).get('matches')


In [ ]:
def generate_answer(question, top_k=3, model='gpt-4o'):
    results = query_from_pinecone(question, top_k=top_k)
    context = '

'.join([r['metadata']['text'] for r in results])
    prompt = f'Answer the question using only the following context:
{context}

Question: {question}
Answer:'
    response = client.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


In [ ]:
# Example usage
answer = generate_answer('How do I replace a lost medicare card?')
print(answer)
